# Kaggriculture: More Wheat, Smarter Sales

Kaggriculture combines farm production with a shared market. This agent builds
on The 2945 Farm and adds two small improvements: a more productive opening
wheat crop and a stock-aware queue for cash-product sales.

The opening buys five wheat and one extra seed. A worker grows wheat on a
future pasture at (2, 4). Waiting until day three yields three wheat instead
of two. The worker harvests, restores the pasture and delivers the crop before
the original cow placement at callback 95. This cycle needs no extra land
or hired workers.

The sale queue fills unusable cash-sale slots with later executable sales,
tracking projected stock as orders are processed. Purchases, hiring, wheat
and fertilizer order positions remain protected, and the market-order limit
is preserved. The remaining farm policy retains the upstream implementation.

Run all cells to create `submission.tar.gz` and reproduce one complete game.
Code is collapsed by default.


## Local comparison — September 20, 2026

The reacting control is the five-wheat opening with a temporary crop harvested
on day two. Each change played ten complete games: five worlds, both seats.
The combined policy used five different worlds from the individual comparisons.

| Change | Wins / losses / ties | Mean final-money margin | Worst margin |
|---|---:|---:|---:|
| Harvest opening wheat on day three | 10 / 0 / 0 | +27.6 | +27 |
| Fill executable cash-sale slots | 8 / 0 / 2 | +47.8 | 0 |
| Both changes (the exported agent) | 10 / 0 / 0 | +75.4 | +28 |

The combined agent harvested and delivered three opening wheat units and
restored the pasture in all ten games. All games completed with passing
economic conservation and action-limit checks. Two additional ideas produced
only ties with no demonstrated behavioral benefit and are not included.
Including five control mirrors, the complete screen comprised 55 games.

This is a small comparison against one related opponent. Both seats share
world seeds; these are not ten independent worlds. It does not establish
strength against unrelated opponents or a leaderboard improvement. The demo
below repeats one combined-policy game and adds no new qualification evidence.


## Credits and reproducibility

The base is [Thomas Tschinkel's The 2945 Farm](https://www.kaggle.com/code/thomastschinkel/the-2945-farm-96-vs-the-top-10-public-bots).
Its original credits and Apache-2.0 notices remain in the distributed source.
The sale-queue idea was informed by [Arlene's Farming Score V2](https://www.kaggle.com/code/lynnsakurai/farming-score-v2-a-better-approach).
The additional crop and queue implementation is by Dmitrii Gluzdov, with
Codex and DeepSeek assistance. `NOTICE.txt` records attribution and licensing.

The demo embeds the exact [Kaggle game implementation](https://github.com/Kaggle/kaggle-environments/tree/master/kaggle_environments/envs/kaggriculture)
and schema from kaggle-environments 1.32.7, its Apache-2.0 license, and a
byte-verified seed-helper compatibility shim for older host cores. No network
fetch is needed. The manifest pins policy sources, worlds and action streams;
the demonstration checks 720 states, DONE/DONE and 719 callbacks per seat.


In [1]:
EVALUATION_MANIFEST = {'engine_distribution': '1.32.7', 'individual_world_seeds': [29454000, 29454001, 29454002, 29454003, 29454004], 'combined_world_seeds': [29454100, 29454101, 29454102, 29454103, 29454104], 'seats': [0, 1], 'games_per_change': 10, 'opponent_url': 'https://www.kaggle.com/code/dmitriigluzdov/kaggriculture-a-smaller-market-shock', 'opponent_notebook_version': 2, 'opponent_source_sha256': 'd5460fc2e5488e0a340f0e4e795f2709c48b58cff7c204ced3821a4c7dae4555', 'candidate_source_sha256': '218d72b0a5ce40d23657b9bd6a1385f106948a9dd76d49f786d9cf04eba34b7d', 'demo_seed': 29454100, 'demo_candidate_seat': 0, 'demo_rewards': [94976.0, 94933.0], 'demo_action_hash_sequence_sha256': ['9d1caf79e29316809dbcb851fd4556d6b222cc392463a470ccf5ee978be13a6c', 'fc8e02da7431b976e4e49d948763e6f0f2be08f1702ca813188d88e990b1bc06']}
print("Local combined comparison: 10 games, five worlds, both seats; 10 wins.")
print("Source SHA-256:", EVALUATION_MANIFEST["candidate_source_sha256"])


In [2]:
from pathlib import Path
import ast, base64, gzip, hashlib, io, tarfile, tempfile

WORK = Path.cwd()
ARCHIVE = WORK / "submission.tar.gz"
ARCHIVE_BYTES = base64.b64decode('...')
EXPECTED_ARCHIVE_SHA256 = "4717f93fd1f0e789a663a9401c69e0c0c906251a1c00d96cae97aa993e3a9376"
EXPECTED_MAIN_SHA256 = "218d72b0a5ce40d23657b9bd6a1385f106948a9dd76d49f786d9cf04eba34b7d"
EXPECTED_NOTICE_SHA256 = "d7da555cfaf9487b9b4070b1b2d3155450ddbd85e69373be728a018bb8ee62f4"

def sha256(data):
    return hashlib.sha256(data).hexdigest()

assert sha256(ARCHIVE_BYTES) == EXPECTED_ARCHIVE_SHA256
ARCHIVE.write_bytes(ARCHIVE_BYTES)
RUNTIME_CLEANUP = tempfile.TemporaryDirectory(prefix="kg-agent-")
RUNTIME = Path(RUNTIME_CLEANUP.name)
with tarfile.open(ARCHIVE, "r:gz") as bundle:
    assert bundle.getnames() == ["NOTICE.txt", "main.py"]
    notice_bytes = bundle.extractfile("NOTICE.txt").read()
    main_bytes = bundle.extractfile("main.py").read()
assert sha256(main_bytes) == EXPECTED_MAIN_SHA256
assert sha256(notice_bytes) == EXPECTED_NOTICE_SHA256
compile(main_bytes, "main.py", "exec")
ast.parse(main_bytes)
MAIN = RUNTIME / "main.py"
NOTICE = RUNTIME / "NOTICE.txt"
MAIN.write_bytes(main_bytes)
NOTICE.write_bytes(notice_bytes)
print("PASS: byte-exact submission archive and preserved upstream files")
print("archive SHA-256:", EXPECTED_ARCHIVE_SHA256)
print("main.py SHA-256:", EXPECTED_MAIN_SHA256)
print("NOTICE.txt SHA-256:", EXPECTED_NOTICE_SHA256)

CONTROL_BYTES = main_bytes[:862940]  # Preserved, byte-exact upstream prefix.
assert sha256(CONTROL_BYTES) == EVALUATION_MANIFEST["opponent_source_sha256"]
CONTROL = RUNTIME / "original.py"
CONTROL.write_bytes(CONTROL_BYTES)

In [3]:
import base64, contextlib, hashlib, importlib.metadata, importlib.util, io, json, os, sys
from pathlib import Path

ENGINE_SOURCE = base64.b64decode("...")
ENGINE_SCHEMA = base64.b64decode("ewogICJuYW1lIjogImthZ2dyaWN1bHR1cmUiLAogICJ0aXRsZSI6ICJLYWdncmljdWx0dXJlIiwKICAiZGVzY3JpcHRpb24iOiAiQWR2YW5jZWQgZmFybWluZyBzaW11bGF0aW9uOiB0d28gcGxheWVycyBlYWNoIHRlbmQgYSAxMHgxMCBmYXJtIG9mIGZvdXIgNXg1IHF1YWRyYW50cywgZ3Jvd2luZyBjcm9wcywgcmFpc2luZyBhbmltYWxzLCBoaXJpbmcgaGVscCwgYW5kIHRyYWRpbmcgd2l0aCBhIGR5bmFtaWMgbWFya2V0IG92ZXIgb25lIHNlYXNvbi4iLAogICJ2ZXJzaW9uIjogIjAuMS4wIiwKICAiYWdlbnRzIjogWzJdLAogICJjb25maWd1cmF0aW9uIjogewogICAgImVwaXNvZGVTdGVwcyI6IDcyMCwKICAgICJhY3RUaW1lb3V0IjogMSwKICAgICJib2FyZFNpemUiOiB7CiAgICAgICJkZXNjcmlwdGlvbiI6ICJXaWR0aCBhbmQgaGVpZ2h0IChpbiB0aWxlcykgb2YgZWFjaCBwbGF5ZXIncyBzcXVhcmUgZmFybS4gQWR2YW5jZWQgZ2FtZSBleHBlY3RzIDEwIChmb3VyIDV4NSBxdWFkcmFudHMpLiIsCiAgICAgICJ0eXBlIjogImludGVnZXIiLAogICAgICAiZGVmYXVsdCI6IDEwLAogICAgICAibWluaW11bSI6IDQKICAgIH0sCiAgICAic3RhcnRpbmdNb25leSI6IHsKICAgICAgImRlc2NyaXB0aW9uIjogIk1vbmV5IGVhY2ggcGxheWVyIHN0YXJ0cyB3aXRoLiIsCiAgICAgICJ0eXBlIjogImludGVnZXIiLAogICAgICAiZGVmYXVsdCI6IDMwMDAsCiAgICAgICJtaW5pbXVtIjogMAogICAgfSwKICAgICJtYXhNYXJrZXRPcmRlcnNQZXJUdXJuIjogewogICAgICAiZGVzY3JpcHRpb24iOiAiTWF4aW11bSBudW1iZXIgb2YgbWFya2V0IG9yZGVycyBwcm9jZXNzZWQgcGVyIHBsYXllciBwZXIgdHVybi4gRXh0cmEgb3JkZXJzIGJleW9uZCB0aGlzIGxpbWl0IGFyZSBzaWxlbnRseSBkcm9wcGVkLiIsCiAgICAgICJ0eXBlIjogImludGVnZXIiLAogICAgICAiZGVmYXVsdCI6IDEwLAogICAgICAibWluaW11bSI6IDEKICAgIH0sCiAgICAidHVybnNQZXJEYXkiOiB7CiAgICAgICJkZXNjcmlwdGlvbiI6ICJOdW1iZXIgb2YgdHVybnMgdGhhdCBtYWtlIHVwIG9uZSBpbi1nYW1lIGRheS4iLAogICAgICAidHlwZSI6ICJpbnRlZ2VyIiwKICAgICAgImRlZmF1bHQiOiAyNCwKICAgICAgIm1pbmltdW0iOiAxCiAgICB9LAogICAgInNoZWRDYXBhY2l0eSI6IHsKICAgICAgImRlc2NyaXB0aW9uIjogIk1heGltdW0gbnVtYmVyIG9mIG5vbi1zZWVkIGl0ZW1zIHRoZSBzaGVkIGNhbiBob2xkIChvdmVyZmxvdyBhdCBlbmQtb2YtZGF5IGRyb3AgaXMgZGlzY2FyZGVkKS4iLAogICAgICAidHlwZSI6ICJpbnRlZ2VyIiwKICAgICAgImRlZmF1bHQiOiAxMDAsCiAgICAgICJtaW5pbXVtIjogMQogICAgfSwKICAgICJ3ZWVkU3Bhd25DaGFuY2UiOiB7CiAgICAgICJkZXNjcmlwdGlvbiI6ICJQZXItdGlsZSBwcm9iYWJpbGl0eSBvZiBhIHdlZWQgc3Bhd25pbmcgb24gYW4gZW1wdHkgdW5sb2NrZWQgdGlsZSBkdXJpbmcgdGhlIGVuZC1vZi1kYXkgcmVmcmVzaC4iLAogICAgICAidHlwZSI6ICJudW1iZXIiLAogICAgICAiZGVmYXVsdCI6IDAuMDA1LAogICAgICAibWluaW11bSI6IDAKICAgIH0sCiAgICAidG93blNob3BVbmxvY2tJbnRlcnZhbCI6IHsKICAgICAgImRlc2NyaXB0aW9uIjogIkRheXMgYmV0d2VlbiBzdWNjZXNzaXZlIHRvd24gc2hvcCB1bmxvY2tzLiBGaXJzdCBzaG9wIHVubG9ja3Mgb24gZGF5IGVxdWFsIHRvIHRoaXMgdmFsdWUuIFNob3BzIGFyZSBkcmF3biB3aXRoIHJlcGxhY2VtZW50LCBzbyB0aGUgc2FtZSBzaG9wIGNhbiB1bmxvY2sgbW9yZSB0aGFuIG9uY2U7IHVubG9ja2luZyBzdG9wcyBhZnRlciA4IGluc3RhbmNlcy4iLAogICAgICAidHlwZSI6ICJpbnRlZ2VyIiwKICAgICAgImRlZmF1bHQiOiAzLAogICAgICAibWluaW11bSI6IDEKICAgIH0sCiAgICAidG93blNob3BTZWxsSW50ZXJ2YWwiOiB7CiAgICAgICJkZXNjcmlwdGlvbiI6ICJOdW1iZXIgb2YgdHVybnMgYmV0d2VlbiBzdWNjZXNzaXZlIGNvbnN1bXB0aW9uIHRpY2tzIGJ5IGV2ZXJ5IHVubG9ja2VkIHRvd24gc2hvcCBpbnN0YW5jZS4gRWFjaCBpbnN0YW5jZSBwdWxscyBvbmUgb2YgZWFjaCBvZiBpdHMgcHJvZHVjdHMgcGVyIHRpY2sgKHNpbmdsZS1wcm9kdWN0IHNob3BzIHB1bGwgMngpLCBzbyBhIGR1cGxpY2F0ZWQgc2hvcCBjb25zdW1lcyBvbmNlIHBlciBjb3B5LiBUb3RhbCBkZW1hbmQgZ3Jvd3MgbW9ub3RvbmljYWxseSBhcyBtb3JlIHNob3BzIGFyZSB1bmxvY2tlZC4iLAogICAgICAidHlwZSI6ICJpbnRlZ2VyIiwKICAgICAgImRlZmF1bHQiOiA0LAogICAgICAibWluaW11bSI6IDEKICAgIH0sCiAgICAidG93bkNlbnRlclNlbGxJbnRlcnZhbCI6IHsKICAgICAgImRlc2NyaXB0aW9uIjogIk51bWJlciBvZiB0dXJucyBiZXR3ZWVuIHN1Y2Nlc3NpdmUgY29uc3VtcHRpb24gdGlja3MgYnkgdGhlIHRvd24gY2VudGVyIChvbmUgb2YgZXZlcnkgbm9uLWZlcnRpbGl6ZXIgcHJvZHVjdCBwZXIgdGljaykuIEF0IHRoZSBkZWZhdWx0IG9mIDI0IHdpdGggdHVybnNQZXJEYXkgMjQsIHRoZSB0b3duIGNlbnRlciBidXlzIG9uY2UgcGVyIGRheSBhdCBhIGZsYXQgcmF0ZSBmb3IgdGhlIHdob2xlIHNlYXNvbi4iLAogICAgICAidHlwZSI6ICJpbnRlZ2VyIiwKICAgICAgImRlZmF1bHQiOiAyNCwKICAgICAgIm1pbmltdW0iOiAxCiAgICB9LAogICAgInNlZWQiOiB7CiAgICAgICJkZXNjcmlwdGlvbiI6ICJPcHRpb25hbCBpbnB1dCBzZWVkIGZvciBkZXRlcm1pbmlzdGljIGVwaXNvZGUgZ2VuZXJhdGlvbi4gVGhlIGludGVycHJldGVyIGNsZWFycyB0aGlzIGZyb20gY29uZmlndXJhdGlvbiBhZnRlciByZWFkaW5nIGFuZCBzdG9yZXMgdGhlIHJlc29sdmVkIHNlZWQgb24gZW52LmluZm9bJ3NlZWQnXSBzbyBpdCBzdGF5cyBvdXQgb2YgYWdlbnQgb2JzZXJ2YXRpb25zIGJ1dCBpcyBzdGlsbCByZWNvcmRlZCBpbiB0aGUgcmVwbGF5LiIsCiAgICAgICJ0eXBlIjogWyJpbnRlZ2VyIiwgIm51bGwiXSwKICAgICAgImRlZmF1bHQiOiBudWxsCiAgICB9LAogICAgImZhcm1IYW5kQ29zdE11bHQiOiB7CiAgICAgICJkZXNjcmlwdGlvbiI6ICJNdWx0aXBsaWVyIGFwcGxpZWQgdG8gdGhlIEZpYm9uYWNjaSBoaXJlLWNvc3Qgc2VxdWVuY2UgKDEsIDEsIDIsIDMsIDUsIDgsIDEzLCAuLi4pLiBUaGUgbi10aCBoaXJlIG9mIHRoZSBkYXkgY29zdHMgdGhpcyB2YWx1ZSB0aW1lcyBmaWIobikuIiwKICAgICAgInR5cGUiOiAiaW50ZWdlciIsCiAgICAgICJkZWZhdWx0IjogMSwKICAgICAgIm1pbmltdW0iOiAwCiAgICB9LAogICAgIm1hcmtldFBhcmFtcyI6IHsKICAgICAgImRlc2NyaXB0aW9uIjogIlBlci1yZXNvdXJjZSBtYXJrZXQgcHJpY2UtY3VydmUgb3ZlcnJpZGVzLiBTcGFyc2U6IGtleWVkIGJ5IHByb2R1Y3QgbmFtZTsgZWFjaCBlbnRyeSBtYXkgc2V0IGFueSBzdWJzZXQgb2Yge2Jhc2UsIEkwLCBULCBiZWxvd19mdW5jLCBiZWxvd190YXJnZXQsIGFib3ZlX2Z1bmMsIGFib3ZlX3RhcmdldH0uIE1pc3Npbmcga2V5cyAoYW5kIG1pc3NpbmcgcHJvZHVjdHMpIGluaGVyaXQgZnJvbSBNQVJLRVRfUEFSQU1TIGRlZmF1bHRzIGluIGthZ2dyaWN1bHR1cmUucHkuIEZ1bmN0aW9uczogbGluZWFyLCBzcSwgc3FydCwgbG9nLCBsb2cxMCwgaGluZ2UuIGFtcCBpcyBkZXJpdmVkIGFzIHRhcmdldCAqIGJhc2UgLyBmKFQpLiBUaGUgaGluZ2UgZnVuY3Rpb24gaXMgZmxhdC1pc2ggYmVsb3cgVCBhbmQgc3Bpa2VzIGFib3ZlIGl0LCBhbmQgdW5saWtlIHRoZSBvdGhlcnMgaXQgaXMgc2NhbGVkIGJ5IFQgc28gdGhhdCBmKFQpID0gMS4iLAogICAgICAidHlwZSI6ICJvYmplY3QiLAogICAgICAiZGVmYXVsdCI6IHt9CiAgICB9CiAgfSwKICAicmV3YXJkIjogewogICAgImRlc2NyaXB0aW9uIjogIlBsYXllciBtb25leSBhdCBlbmQgb2YgZ2FtZSAoZmluYWwgc2NvcmUpLiIsCiAgICAidHlwZSI6ICJudW1iZXIiLAogICAgImRlZmF1bHQiOiAwCiAgfSwKICAib2JzZXJ2YXRpb24iOiB7CiAgICAicGxheWVyIjogewogICAgICAiZGVzY3JpcHRpb24iOiAiUGxheWVyIElEICgwIG9yIDEpLiIsCiAgICAgICJ0eXBlIjogImludGVnZXIiLAogICAgICAiZGVmYXVsdCI6IDAKICAgIH0sCiAgICAiZmFybXMiOiB7CiAgICAgICJkZXNjcmlwdGlvbiI6ICJQZXItcGxheWVyIHB1YmxpYyBmYXJtIHN0YXRlIHZpc2libGUgdG8gYWxsIGFnZW50cyAodGlsZXMsIG1vbmV5LCBmYXJtZXIvaGFuZCBwb3NpdGlvbnMsIHVubG9ja2VkIHF1YWRyYW50cywgaGlyZSBjb3VudCkuIEluZGV4ZWQgYnkgcGxheWVyIGlkLiBPcHBvbmVudCBzaGVkIGFuZCBwZXItZmFybWVyIGludmVudG9yaWVzIGFyZSBOT1QgaW5jbHVkZWQgaGVyZS4iLAogICAgICAidHlwZSI6ICJhcnJheSIsCiAgICAgICJzaGFyZWQiOiB0cnVlLAogICAgICAiZGVmYXVsdCI6IFtdCiAgICB9LAogICAgInByaXZhdGUiOiB7CiAgICAgICJkZXNjcmlwdGlvbiI6ICJQZXItYWdlbnQgcHJpdmF0ZSBzdGF0ZSBmb3IgVEhJUyBwbGF5ZXIgb25seTogc2hlZCBpbnZlbnRvcnksIHBlci1mYXJtZXIgaW52ZW50b3JpZXMsIHNlZWQgY291bnRzLiBOb3Qgc2hhcmVkLiIsCiAgICAgICJ0eXBlIjogIm9iamVjdCIsCiAgICAgICJzaGFyZWQiOiBmYWxzZSwKICAgICAgImRlZmF1bHQiOiB7fQogICAgfSwKICAgICJtYXJrZXQiOiB7CiAgICAgICJkZXNjcmlwdGlvbiI6ICJTaGFyZWQgbWFya2V0IHN0YXRlOiBjdXJyZW50IHBlci1wcm9kdWN0IGludmVudG9yeSBhbmQgcHJpY2UuIiwKICAgICAgInR5cGUiOiAib2JqZWN0IiwKICAgICAgInNoYXJlZCI6IHRydWUsCiAgICAgICJkZWZhdWx0Ijoge30KICAgIH0sCiAgICAidG93biI6IHsKICAgICAgImRlc2NyaXB0aW9uIjogIlNoYXJlZCB0b3duIHN0YXRlOiBsaXN0IG9mIHVubG9ja2VkIHNob3AgbmFtZXMuIEVhY2ggdW5sb2NrZWQgc2hvcCBnZW5lcmF0ZXMgcGVyLXRpY2sgZGVtYW5kIGZvciBpdHMgcHJvZHVjdHMuIiwKICAgICAgInR5cGUiOiAib2JqZWN0IiwKICAgICAgInNoYXJlZCI6IHRydWUsCiAgICAgICJkZWZhdWx0Ijoge30KICAgIH0sCiAgICAiZGF5IjogewogICAgICAiZGVzY3JpcHRpb24iOiAiQ3VycmVudCBpbi1nYW1lIGRheSAoMC1pbmRleGVkKS4iLAogICAgICAidHlwZSI6ICJpbnRlZ2VyIiwKICAgICAgInNoYXJlZCI6IHRydWUsCiAgICAgICJkZWZhdWx0IjogMAogICAgfSwKICAgICJob3VyIjogewogICAgICAiZGVzY3JpcHRpb24iOiAiQ3VycmVudCB0dXJuIHdpdGhpbiB0aGUgZGF5ICgwLWluZGV4ZWQpLiIsCiAgICAgICJ0eXBlIjogImludGVnZXIiLAogICAgICAic2hhcmVkIjogdHJ1ZSwKICAgICAgImRlZmF1bHQiOiAwCiAgICB9LAogICAgInJlbWFpbmluZ092ZXJhZ2VUaW1lIjogNjAKICB9LAogICJhY3Rpb24iOiB7CiAgICAiZGVzY3JpcHRpb24iOiAiUGVyLXR1cm4gYWN0aW9uIG9mIHRoZSBmb3JtIHtcImZhcm1lclwiOiBbb3AsIC4uLmFyZ3NdLCBcImhhbmRzXCI6IFtbb3AsIC4uLmFyZ3NdLCAuLi5dLCBcIm1hcmtldFwiOiBbW29wLCAuLi5hcmdzXSwgLi4uXX0uIEZhcm1lci9oYW5kIG9wczogTk9SVEgsIFNPVVRILCBFQVNULCBXRVNULCBQQVNTLCBQSUNLVVAgPGl0ZW0+IFtuXSwgUExBTlQgPGNyb3A+LCBXQVRFUiwgSEFSVkVTVCwgRkVSVElMSVpFLCBCVUlMRF9DT09QLCBCVUlMRF9QQVNUVVJFLCBESUcsIFBMQUNFIDxpdGVtPiBbbl0sIEZFRUQsIENPTExFQ1RfRkVSVElMSVpFUiwgQ0FSRS4gTWFya2V0IG9wczogQlVZX1NFRUQgPGNyb3A+IDxuPiwgQlVZX1BST0RVQ1QgPGl0ZW0+IDxuPiwgQlVZX0FOSU1BTCA8YW5pbWFsPiA8bj4sIFNFTEwgPGl0ZW0+IDxuPiwgSElSRSwgQlVZX0xBTkQuIiwKICAgICJ0eXBlIjogIm9iamVjdCIsCiAgICAiZGVmYXVsdCI6IHsgImZhcm1lciI6IFsiUEFTUyJdLCAiaGFuZHMiOiBbXSwgIm1hcmtldCI6IFtdIH0KICB9LAogICJzdGF0dXMiOiB7CiAgICAiZGVmYXVsdHMiOiBbIkFDVElWRSIsICJBQ1RJVkUiXQogIH0KfQo=")
ENGINE_LICENSE = base64.b64decode("ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgQXBhY2hlIExpY2Vuc2UKICAgICAgICAgICAgICAgICAgICAgICAgICAgVmVyc2lvbiAyLjAsIEphbnVhcnkgMjAwNAogICAgICAgICAgICAgICAgICAgICAgICBodHRwOi8vd3d3LmFwYWNoZS5vcmcvbGljZW5zZXMvCgogICBURVJNUyBBTkQgQ09ORElUSU9OUyBGT1IgVVNFLCBSRVBST0RVQ1RJT04sIEFORCBESVNUUklCVVRJT04KCiAgIDEuIERlZmluaXRpb25zLgoKICAgICAgIkxpY2Vuc2UiIHNoYWxsIG1lYW4gdGhlIHRlcm1zIGFuZCBjb25kaXRpb25zIGZvciB1c2UsIHJlcHJvZHVjdGlvbiwKICAgICAgYW5kIGRpc3RyaWJ1dGlvbiBhcyBkZWZpbmVkIGJ5IFNlY3Rpb25zIDEgdGhyb3VnaCA5IG9mIHRoaXMgZG9jdW1lbnQuCgogICAgICAiTGljZW5zb3IiIHNoYWxsIG1lYW4gdGhlIGNvcHlyaWdodCBvd25lciBvciBlbnRpdHkgYXV0aG9yaXplZCBieQogICAgICB0aGUgY29weXJpZ2h0IG93bmVyIHRoYXQgaXMgZ3JhbnRpbmcgdGhlIExpY2Vuc2UuCgogICAgICAiTGVnYWwgRW50aXR5IiBzaGFsbCBtZWFuIHRoZSB1bmlvbiBvZiB0aGUgYWN0aW5nIGVudGl0eSBhbmQgYWxsCiAgICAgIG90aGVyIGVudGl0aWVzIHRoYXQgY29udHJvbCwgYXJlIGNvbnRyb2xsZWQgYnksIG9yIGFyZSB1bmRlciBjb21tb24KICAgICAgY29udHJvbCB3aXRoIHRoYXQgZW50aXR5LiBGb3IgdGhlIHB1cnBvc2VzIG9mIHRoaXMgZGVmaW5pdGlvbiwKICAgICAgImNvbnRyb2wiIG1lYW5zIChpKSB0aGUgcG93ZXIsIGRpcmVjdCBvciBpbmRpcmVjdCwgdG8gY2F1c2UgdGhlCiAgICAgIGRpcmVjdGlvbiBvciBtYW5hZ2VtZW50IG9mIHN1Y2ggZW50aXR5LCB3aGV0aGVyIGJ5IGNvbnRyYWN0IG9yCiAgICAgIG90aGVyd2lzZSwgb3IgKGlpKSBvd25lcnNoaXAgb2YgZmlmdHkgcGVyY2VudCAoNTAlKSBvciBtb3JlIG9mIHRoZQogICAgICBvdXRzdGFuZGluZyBzaGFyZXMsIG9yIChpaWkpIGJlbmVmaWNpYWwgb3duZXJzaGlwIG9mIHN1Y2ggZW50aXR5LgoKICAgICAgIllvdSIgKG9yICJZb3VyIikgc2hhbGwgbWVhbiBhbiBpbmRpdmlkdWFsIG9yIExlZ2FsIEVudGl0eQogICAgICBleGVyY2lzaW5nIHBlcm1pc3Npb25zIGdyYW50ZWQgYnkgdGhpcyBMaWNlbnNlLgoKICAgICAgIlNvdXJjZSIgZm9ybSBzaGFsbCBtZWFuIHRoZSBwcmVmZXJyZWQgZm9ybSBmb3IgbWFraW5nIG1vZGlmaWNhdGlvbnMsCiAgICAgIGluY2x1ZGluZyBidXQgbm90IGxpbWl0ZWQgdG8gc29mdHdhcmUgc291cmNlIGNvZGUsIGRvY3VtZW50YXRpb24KICAgICAgc291cmNlLCBhbmQgY29uZmlndXJhdGlvbiBmaWxlcy4KCiAgICAgICJPYmplY3QiIGZvcm0gc2hhbGwgbWVhbiBhbnkgZm9ybSByZXN1bHRpbmcgZnJvbSBtZWNoYW5pY2FsCiAgICAgIHRyYW5zZm9ybWF0aW9uIG9yIHRyYW5zbGF0aW9uIG9mIGEgU291cmNlIGZvcm0sIGluY2x1ZGluZyBidXQKICAgICAgbm90IGxpbWl0ZWQgdG8gY29tcGlsZWQgb2JqZWN0IGNvZGUsIGdlbmVyYXRlZCBkb2N1bWVudGF0aW9uLAogICAgICBhbmQgY29udmVyc2lvbnMgdG8gb3RoZXIgbWVkaWEgdHlwZXMuCgogICAgICAiV29yayIgc2hhbGwgbWVhbiB0aGUgd29yayBvZiBhdXRob3JzaGlwLCB3aGV0aGVyIGluIFNvdXJjZSBvcgogICAgICBPYmplY3QgZm9ybSwgbWFkZSBhdmFpbGFibGUgdW5kZXIgdGhlIExpY2Vuc2UsIGFzIGluZGljYXRlZCBieSBhCiAgICAgIGNvcHlyaWdodCBub3RpY2UgdGhhdCBpcyBpbmNsdWRlZCBpbiBvciBhdHRhY2hlZCB0byB0aGUgd29yawogICAgICAoYW4gZXhhbXBsZSBpcyBwcm92aWRlZCBpbiB0aGUgQXBwZW5kaXggYmVsb3cpLgoKICAgICAgIkRlcml2YXRpdmUgV29ya3MiIHNoYWxsIG1lYW4gYW55IHdvcmssIHdoZXRoZXIgaW4gU291cmNlIG9yIE9iamVjdAogICAgICBmb3JtLCB0aGF0IGlzIGJhc2VkIG9uIChvciBkZXJpdmVkIGZyb20pIHRoZSBXb3JrIGFuZCBmb3Igd2hpY2ggdGhlCiAgICAgIGVkaXRvcmlhbCByZXZpc2lvbnMsIGFubm90YXRpb25zLCBlbGFib3JhdGlvbnMsIG9yIG90aGVyIG1vZGlmaWNhdGlvbnMKICAgICAgcmVwcmVzZW50LCBhcyBhIHdob2xlLCBhbiBvcmlnaW5hbCB3b3JrIG9mIGF1dGhvcnNoaXAuIEZvciB0aGUgcHVycG9zZXMKICAgICAgb2YgdGhpcyBMaWNlbnNlLCBEZXJpdmF0aXZlIFdvcmtzIHNoYWxsIG5vdCBpbmNsdWRlIHdvcmtzIHRoYXQgcmVtYWluCiAgICAgIHNlcGFyYWJsZSBmcm9tLCBvciBtZXJlbHkgbGluayAob3IgYmluZCBieSBuYW1lKSB0byB0aGUgaW50ZXJmYWNlcyBvZiwKICAgICAgdGhlIFdvcmsgYW5kIERlcml2YXRpdmUgV29ya3MgdGhlcmVvZi4KCiAgICAgICJDb250cmlidXRpb24iIHNoYWxsIG1lYW4gYW55IHdvcmsgb2YgYXV0aG9yc2hpcCwgaW5jbHVkaW5nCiAgICAgIHRoZSBvcmlnaW5hbCB2ZXJzaW9uIG9mIHRoZSBXb3JrIGFuZCBhbnkgbW9kaWZpY2F0aW9ucyBvciBhZGRpdGlvbnMKICAgICAgdG8gdGhhdCBXb3JrIG9yIERlcml2YXRpdmUgV29ya3MgdGhlcmVvZiwgdGhhdCBpcyBpbnRlbnRpb25hbGx5CiAgICAgIHN1Ym1pdHRlZCB0byBMaWNlbnNvciBmb3IgaW5jbHVzaW9uIGluIHRoZSBXb3JrIGJ5IHRoZSBjb3B5cmlnaHQgb3duZXIKICAgICAgb3IgYnkgYW4gaW5kaXZpZHVhbCBvciBMZWdhbCBFbnRpdHkgYXV0aG9yaXplZCB0byBzdWJtaXQgb24gYmVoYWxmIG9mCiAgICAgIHRoZSBjb3B5cmlnaHQgb3duZXIuIEZvciB0aGUgcHVycG9zZXMgb2YgdGhpcyBkZWZpbml0aW9uLCAic3VibWl0dGVkIgogICAgICBtZWFucyBhbnkgZm9ybSBvZiBlbGVjdHJvbmljLCB2ZXJiYWwsIG9yIHdyaXR0ZW4gY29tbXVuaWNhdGlvbiBzZW50CiAgICAgIHRvIHRoZSBMaWNlbnNvciBvciBpdHMgcmVwcmVzZW50YXRpdmVzLCBpbmNsdWRpbmcgYnV0IG5vdCBsaW1pdGVkIHRvCiAgICAgIGNvbW11bmljYXRpb24gb24gZWxlY3Ryb25pYyBtYWlsaW5nIGxpc3RzLCBzb3VyY2UgY29kZSBjb250cm9sIHN5c3RlbXMsCiAgICAgIGFuZCBpc3N1ZSB0cmFja2luZyBzeXN0ZW1zIHRoYXQgYXJlIG1hbmFnZWQgYnksIG9yIG9uIGJlaGFsZiBvZiwgdGhlCiAgICAgIExpY2Vuc29yIGZvciB0aGUgcHVycG9zZSBvZiBkaXNjdXNzaW5nIGFuZCBpbXByb3ZpbmcgdGhlIFdvcmssIGJ1dAogICAgICBleGNsdWRpbmcgY29tbXVuaWNhdGlvbiB0aGF0IGlzIGNvbnNwaWN1b3VzbHkgbWFya2VkIG9yIG90aGVyd2lzZQogICAgICBkZXNpZ25hdGVkIGluIHdyaXRpbmcgYnkgdGhlIGNvcHlyaWdodCBvd25lciBhcyAiTm90IGEgQ29udHJpYnV0aW9uLiIKCiAgICAgICJDb250cmlidXRvciIgc2hhbGwgbWVhbiBMaWNlbnNvciBhbmQgYW55IGluZGl2aWR1YWwgb3IgTGVnYWwgRW50aXR5CiAgICAgIG9uIGJlaGFsZiBvZiB3aG9tIGEgQ29udHJpYnV0aW9uIGhhcyBiZWVuIHJlY2VpdmVkIGJ5IExpY2Vuc29yIGFuZAogICAgICBzdWJzZXF1ZW50bHkgaW5jb3Jwb3JhdGVkIHdpdGhpbiB0aGUgV29yay4KCiAgIDIuIEdyYW50IG9mIENvcHlyaWdodCBMaWNlbnNlLiBTdWJqZWN0IHRvIHRoZSB0ZXJtcyBhbmQgY29uZGl0aW9ucyBvZgogICAgICB0aGlzIExpY2Vuc2UsIGVhY2ggQ29udHJpYnV0b3IgaGVyZWJ5IGdyYW50cyB0byBZb3UgYSBwZXJwZXR1YWwsCiAgICAgIHdvcmxkd2lkZSwgbm9uLWV4Y2x1c2l2ZSwgbm8tY2hhcmdlLCByb3lhbHR5LWZyZWUsIGlycmV2b2NhYmxlCiAgICAgIGNvcHlyaWdodCBsaWNlbnNlIHRvIHJlcHJvZHVjZSwgcHJlcGFyZSBEZXJpdmF0aXZlIFdvcmtzIG9mLAogICAgICBwdWJsaWNseSBkaXNwbGF5LCBwdWJsaWNseSBwZXJmb3JtLCBzdWJsaWNlbnNlLCBhbmQgZGlzdHJpYnV0ZSB0aGUKICAgICAgV29yayBhbmQgc3VjaCBEZXJpdmF0aXZlIFdvcmtzIGluIFNvdXJjZSBvciBPYmplY3QgZm9ybS4KCiAgIDMuIEdyYW50IG9mIFBhdGVudCBMaWNlbnNlLiBTdWJqZWN0IHRvIHRoZSB0ZXJtcyBhbmQgY29uZGl0aW9ucyBvZgogICAgICB0aGlzIExpY2Vuc2UsIGVhY2ggQ29udHJpYnV0b3IgaGVyZWJ5IGdyYW50cyB0byBZb3UgYSBwZXJwZXR1YWwsCiAgICAgIHdvcmxkd2lkZSwgbm9uLWV4Y2x1c2l2ZSwgbm8tY2hhcmdlLCByb3lhbHR5LWZyZWUsIGlycmV2b2NhYmxlCiAgICAgIChleGNlcHQgYXMgc3RhdGVkIGluIHRoaXMgc2VjdGlvbikgcGF0ZW50IGxpY2Vuc2UgdG8gbWFrZSwgaGF2ZSBtYWRlLAogICAgICB1c2UsIG9mZmVyIHRvIHNlbGwsIHNlbGwsIGltcG9ydCwgYW5kIG90aGVyd2lzZSB0cmFuc2ZlciB0aGUgV29yaywKICAgICAgd2hlcmUgc3VjaCBsaWNlbnNlIGFwcGxpZXMgb25seSB0byB0aG9zZSBwYXRlbnQgY2xhaW1zIGxpY2Vuc2FibGUKICAgICAgYnkgc3VjaCBDb250cmlidXRvciB0aGF0IGFyZSBuZWNlc3NhcmlseSBpbmZyaW5nZWQgYnkgdGhlaXIKICAgICAgQ29udHJpYnV0aW9uKHMpIGFsb25lIG9yIGJ5IGNvbWJpbmF0aW9uIG9mIHRoZWlyIENvbnRyaWJ1dGlvbihzKQogICAgICB3aXRoIHRoZSBXb3JrIHRvIHdoaWNoIHN1Y2ggQ29udHJpYnV0aW9uKHMpIHdhcyBzdWJtaXR0ZWQuIElmIFlvdQogICAgICBpbnN0aXR1dGUgcGF0ZW50IGxpdGlnYXRpb24gYWdhaW5zdCBhbnkgZW50aXR5IChpbmNsdWRpbmcgYQogICAgICBjcm9zcy1jbGFpbSBvciBjb3VudGVyY2xhaW0gaW4gYSBsYXdzdWl0KSBhbGxlZ2luZyB0aGF0IHRoZSBXb3JrCiAgICAgIG9yIGEgQ29udHJpYnV0aW9uIGluY29ycG9yYXRlZCB3aXRoaW4gdGhlIFdvcmsgY29uc3RpdHV0ZXMgZGlyZWN0CiAgICAgIG9yIGNvbnRyaWJ1dG9yeSBwYXRlbnQgaW5mcmluZ2VtZW50LCB0aGVuIGFueSBwYXRlbnQgbGljZW5zZXMKICAgICAgZ3JhbnRlZCB0byBZb3UgdW5kZXIgdGhpcyBMaWNlbnNlIGZvciB0aGF0IFdvcmsgc2hhbGwgdGVybWluYXRlCiAgICAgIGFzIG9mIHRoZSBkYXRlIHN1Y2ggbGl0aWdhdGlvbiBpcyBmaWxlZC4KCiAgIDQuIFJlZGlzdHJpYnV0aW9uLiBZb3UgbWF5IHJlcHJvZHVjZSBhbmQgZGlzdHJpYnV0ZSBjb3BpZXMgb2YgdGhlCiAgICAgIFdvcmsgb3IgRGVyaXZhdGl2ZSBXb3JrcyB0aGVyZW9mIGluIGFueSBtZWRpdW0sIHdpdGggb3Igd2l0aG91dAogICAgICBtb2RpZmljYXRpb25zLCBhbmQgaW4gU291cmNlIG9yIE9iamVjdCBmb3JtLCBwcm92aWRlZCB0aGF0IFlvdQogICAgICBtZWV0IHRoZSBmb2xsb3dpbmcgY29uZGl0aW9uczoKCiAgICAgIChhKSBZb3UgbXVzdCBnaXZlIGFueSBvdGhlciByZWNpcGllbnRzIG9mIHRoZSBXb3JrIG9yCiAgICAgICAgICBEZXJpdmF0aXZlIFdvcmtzIGEgY29weSBvZiB0aGlzIExpY2Vuc2U7IGFuZAoKICAgICAgKGIpIFlvdSBtdXN0IGNhdXNlIGFueSBtb2RpZmllZCBmaWxlcyB0byBjYXJyeSBwcm9taW5lbnQgbm90aWNlcwogICAgICAgICAgc3RhdGluZyB0aGF0IFlvdSBjaGFuZ2VkIHRoZSBmaWxlczsgYW5kCgogICAgICAoYykgWW91IG11c3QgcmV0YWluLCBpbiB0aGUgU291cmNlIGZvcm0gb2YgYW55IERlcml2YXRpdmUgV29ya3MKICAgICAgICAgIHRoYXQgWW91IGRpc3RyaWJ1dGUsIGFsbCBjb3B5cmlnaHQsIHBhdGVudCwgdHJhZGVtYXJrLCBhbmQKICAgICAgICAgIGF0dHJpYnV0aW9uIG5vdGljZXMgZnJvbSB0aGUgU291cmNlIGZvcm0gb2YgdGhlIFdvcmssCiAgICAgICAgICBleGNsdWRpbmcgdGhvc2Ugbm90aWNlcyB0aGF0IGRvIG5vdCBwZXJ0YWluIHRvIGFueSBwYXJ0IG9mCiAgICAgICAgICB0aGUgRGVyaXZhdGl2ZSBXb3JrczsgYW5kCgogICAgICAoZCkgSWYgdGhlIFdvcmsgaW5jbHVkZXMgYSAiTk9USUNFIiB0ZXh0IGZpbGUgYXMgcGFydCBvZiBpdHMKICAgICAgICAgIGRpc3RyaWJ1dGlvbiwgdGhlbiBhbnkgRGVyaXZhdGl2ZSBXb3JrcyB0aGF0IFlvdSBkaXN0cmlidXRlIG11c3QKICAgICAgICAgIGluY2x1ZGUgYSByZWFkYWJsZSBjb3B5IG9mIHRoZSBhdHRyaWJ1dGlvbiBub3RpY2VzIGNvbnRhaW5lZAogICAgICAgICAgd2l0aGluIHN1Y2ggTk9USUNFIGZpbGUsIGV4Y2x1ZGluZyB0aG9zZSBub3RpY2VzIHRoYXQgZG8gbm90CiAgICAgICAgICBwZXJ0YWluIHRvIGFueSBwYXJ0IG9mIHRoZSBEZXJpdmF0aXZlIFdvcmtzLCBpbiBhdCBsZWFzdCBvbmUKICAgICAgICAgIG9mIHRoZSBmb2xsb3dpbmcgcGxhY2VzOiB3aXRoaW4gYSBOT1RJQ0UgdGV4dCBmaWxlIGRpc3RyaWJ1dGVkCiAgICAgICAgICBhcyBwYXJ0IG9mIHRoZSBEZXJpdmF0aXZlIFdvcmtzOyB3aXRoaW4gdGhlIFNvdXJjZSBmb3JtIG9yCiAgICAgICAgICBkb2N1bWVudGF0aW9uLCBpZiBwcm92aWRlZCBhbG9uZyB3aXRoIHRoZSBEZXJpdmF0aXZlIFdvcmtzOyBvciwKICAgICAgICAgIHdpdGhpbiBhIGRpc3BsYXkgZ2VuZXJhdGVkIGJ5IHRoZSBEZXJpdmF0aXZlIFdvcmtzLCBpZiBhbmQKICAgICAgICAgIHdoZXJldmVyIHN1Y2ggdGhpcmQtcGFydHkgbm90aWNlcyBub3JtYWxseSBhcHBlYXIuIFRoZSBjb250ZW50cwogICAgICAgICAgb2YgdGhlIE5PVElDRSBmaWxlIGFyZSBmb3IgaW5mb3JtYXRpb25hbCBwdXJwb3NlcyBvbmx5IGFuZAogICAgICAgICAgZG8gbm90IG1vZGlmeSB0aGUgTGljZW5zZS4gWW91IG1heSBhZGQgWW91ciBvd24gYXR0cmlidXRpb24KICAgICAgICAgIG5vdGljZXMgd2l0aGluIERlcml2YXRpdmUgV29ya3MgdGhhdCBZb3UgZGlzdHJpYnV0ZSwgYWxvbmdzaWRlCiAgICAgICAgICBvciBhcyBhbiBhZGRlbmR1bSB0byB0aGUgTk9USUNFIHRleHQgZnJvbSB0aGUgV29yaywgcHJvdmlkZWQKICAgICAgICAgIHRoYXQgc3VjaCBhZGRpdGlvbmFsIGF0dHJpYnV0aW9uIG5vdGljZXMgY2Fubm90IGJlIGNvbnN0cnVlZAogICAgICAgICAgYXMgbW9kaWZ5aW5nIHRoZSBMaWNlbnNlLgoKICAgICAgWW91IG1heSBhZGQgWW91ciBvd24gY29weXJpZ2h0IHN0YXRlbWVudCB0byBZb3VyIG1vZGlmaWNhdGlvbnMgYW5kCiAgICAgIG1heSBwcm92aWRlIGFkZGl0aW9uYWwgb3IgZGlmZmVyZW50IGxpY2Vuc2UgdGVybXMgYW5kIGNvbmRpdGlvbnMKICAgICAgZm9yIHVzZSwgcmVwcm9kdWN0aW9uLCBvciBkaXN0cmlidXRpb24gb2YgWW91ciBtb2RpZmljYXRpb25zLCBvcgogICAgICBmb3IgYW55IHN1Y2ggRGVyaXZhdGl2ZSBXb3JrcyBhcyBhIHdob2xlLCBwcm92aWRlZCBZb3VyIHVzZSwKICAgICAgcmVwcm9kdWN0aW9uLCBhbmQgZGlzdHJpYnV0aW9uIG9mIHRoZSBXb3JrIG90aGVyd2lzZSBjb21wbGllcyB3aXRoCiAgICAgIHRoZSBjb25kaXRpb25zIHN0YXRlZCBpbiB0aGlzIExpY2Vuc2UuCgogICA1LiBTdWJtaXNzaW9uIG9mIENvbnRyaWJ1dGlvbnMuIFVubGVzcyBZb3UgZXhwbGljaXRseSBzdGF0ZSBvdGhlcndpc2UsCiAgICAgIGFueSBDb250cmlidXRpb24gaW50ZW50aW9uYWxseSBzdWJtaXR0ZWQgZm9yIGluY2x1c2lvbiBpbiB0aGUgV29yawogICAgICBieSBZb3UgdG8gdGhlIExpY2Vuc29yIHNoYWxsIGJlIHVuZGVyIHRoZSB0ZXJtcyBhbmQgY29uZGl0aW9ucyBvZgogICAgICB0aGlzIExpY2Vuc2UsIHdpdGhvdXQgYW55IGFkZGl0aW9uYWwgdGVybXMgb3IgY29uZGl0aW9ucy4KICAgICAgTm90d2l0aHN0YW5kaW5nIHRoZSBhYm92ZSwgbm90aGluZyBoZXJlaW4gc2hhbGwgc3VwZXJzZWRlIG9yIG1vZGlmeQogICAgICB0aGUgdGVybXMgb2YgYW55IHNlcGFyYXRlIGxpY2Vuc2UgYWdyZWVtZW50IHlvdSBtYXkgaGF2ZSBleGVjdXRlZAogICAgICB3aXRoIExpY2Vuc29yIHJlZ2FyZGluZyBzdWNoIENvbnRyaWJ1dGlvbnMuCgogICA2LiBUcmFkZW1hcmtzLiBUaGlzIExpY2Vuc2UgZG9lcyBub3QgZ3JhbnQgcGVybWlzc2lvbiB0byB1c2UgdGhlIHRyYWRlCiAgICAgIG5hbWVzLCB0cmFkZW1hcmtzLCBzZXJ2aWNlIG1hcmtzLCBvciBwcm9kdWN0IG5hbWVzIG9mIHRoZSBMaWNlbnNvciwKICAgICAgZXhjZXB0IGFzIHJlcXVpcmVkIGZvciByZWFzb25hYmxlIGFuZCBjdXN0b21hcnkgdXNlIGluIGRlc2NyaWJpbmcgdGhlCiAgICAgIG9yaWdpbiBvZiB0aGUgV29yayBhbmQgcmVwcm9kdWNpbmcgdGhlIGNvbnRlbnQgb2YgdGhlIE5PVElDRSBmaWxlLgoKICAgNy4gRGlzY2xhaW1lciBvZiBXYXJyYW50eS4gVW5sZXNzIHJlcXVpcmVkIGJ5IGFwcGxpY2FibGUgbGF3IG9yCiAgICAgIGFncmVlZCB0byBpbiB3cml0aW5nLCBMaWNlbnNvciBwcm92aWRlcyB0aGUgV29yayAoYW5kIGVhY2gKICAgICAgQ29udHJpYnV0b3IgcHJvdmlkZXMgaXRzIENvbnRyaWJ1dGlvbnMpIG9uIGFuICJBUyBJUyIgQkFTSVMsCiAgICAgIFdJVEhPVVQgV0FSUkFOVElFUyBPUiBDT05ESVRJT05TIE9GIEFOWSBLSU5ELCBlaXRoZXIgZXhwcmVzcyBvcgogICAgICBpbXBsaWVkLCBpbmNsdWRpbmcsIHdpdGhvdXQgbGltaXRhdGlvbiwgYW55IHdhcnJhbnRpZXMgb3IgY29uZGl0aW9ucwogICAgICBvZiBUSVRMRSwgTk9OLUlORlJJTkdFTUVOVCwgTUVSQ0hBTlRBQklMSVRZLCBvciBGSVRORVNTIEZPUiBBCiAgICAgIFBBUlRJQ1VMQVIgUFVSUE9TRS4gWW91IGFyZSBzb2xlbHkgcmVzcG9uc2libGUgZm9yIGRldGVybWluaW5nIHRoZQogICAgICBhcHByb3ByaWF0ZW5lc3Mgb2YgdXNpbmcgb3IgcmVkaXN0cmlidXRpbmcgdGhlIFdvcmsgYW5kIGFzc3VtZSBhbnkKICAgICAgcmlza3MgYXNzb2NpYXRlZCB3aXRoIFlvdXIgZXhlcmNpc2Ugb2YgcGVybWlzc2lvbnMgdW5kZXIgdGhpcyBMaWNlbnNlLgoKICAgOC4gTGltaXRhdGlvbiBvZiBMaWFiaWxpdHkuIEluIG5vIGV2ZW50IGFuZCB1bmRlciBubyBsZWdhbCB0aGVvcnksCiAgICAgIHdoZXRoZXIgaW4gdG9ydCAoaW5jbHVkaW5nIG5lZ2xpZ2VuY2UpLCBjb250cmFjdCwgb3Igb3RoZXJ3aXNlLAogICAgICB1bmxlc3MgcmVxdWlyZWQgYnkgYXBwbGljYWJsZSBsYXcgKHN1Y2ggYXMgZGVsaWJlcmF0ZSBhbmQgZ3Jvc3NseQogICAgICBuZWdsaWdlbnQgYWN0cykgb3IgYWdyZWVkIHRvIGluIHdyaXRpbmcsIHNoYWxsIGFueSBDb250cmlidXRvciBiZQogICAgICBsaWFibGUgdG8gWW91IGZvciBkYW1hZ2VzLCBpbmNsdWRpbmcgYW55IGRpcmVjdCwgaW5kaXJlY3QsIHNwZWNpYWwsCiAgICAgIGluY2lkZW50YWwsIG9yIGNvbnNlcXVlbnRpYWwgZGFtYWdlcyBvZiBhbnkgY2hhcmFjdGVyIGFyaXNpbmcgYXMgYQogICAgICByZXN1bHQgb2YgdGhpcyBMaWNlbnNlIG9yIG91dCBvZiB0aGUgdXNlIG9yIGluYWJpbGl0eSB0byB1c2UgdGhlCiAgICAgIFdvcmsgKGluY2x1ZGluZyBidXQgbm90IGxpbWl0ZWQgdG8gZGFtYWdlcyBmb3IgbG9zcyBvZiBnb29kd2lsbCwKICAgICAgd29yayBzdG9wcGFnZSwgY29tcHV0ZXIgZmFpbHVyZSBvciBtYWxmdW5jdGlvbiwgb3IgYW55IGFuZCBhbGwKICAgICAgb3RoZXIgY29tbWVyY2lhbCBkYW1hZ2VzIG9yIGxvc3NlcyksIGV2ZW4gaWYgc3VjaCBDb250cmlidXRvcgogICAgICBoYXMgYmVlbiBhZHZpc2VkIG9mIHRoZSBwb3NzaWJpbGl0eSBvZiBzdWNoIGRhbWFnZXMuCgogICA5LiBBY2NlcHRpbmcgV2FycmFudHkgb3IgQWRkaXRpb25hbCBMaWFiaWxpdHkuIFdoaWxlIHJlZGlzdHJpYnV0aW5nCiAgICAgIHRoZSBXb3JrIG9yIERlcml2YXRpdmUgV29ya3MgdGhlcmVvZiwgWW91IG1heSBjaG9vc2UgdG8gb2ZmZXIsCiAgICAgIGFuZCBjaGFyZ2UgYSBmZWUgZm9yLCBhY2NlcHRhbmNlIG9mIHN1cHBvcnQsIHdhcnJhbnR5LCBpbmRlbW5pdHksCiAgICAgIG9yIG90aGVyIGxpYWJpbGl0eSBvYmxpZ2F0aW9ucyBhbmQvb3IgcmlnaHRzIGNvbnNpc3RlbnQgd2l0aCB0aGlzCiAgICAgIExpY2Vuc2UuIEhvd2V2ZXIsIGluIGFjY2VwdGluZyBzdWNoIG9ibGlnYXRpb25zLCBZb3UgbWF5IGFjdCBvbmx5CiAgICAgIG9uIFlvdXIgb3duIGJlaGFsZiBhbmQgb24gWW91ciBzb2xlIHJlc3BvbnNpYmlsaXR5LCBub3Qgb24gYmVoYWxmCiAgICAgIG9mIGFueSBvdGhlciBDb250cmlidXRvciwgYW5kIG9ubHkgaWYgWW91IGFncmVlIHRvIGluZGVtbmlmeSwKICAgICAgZGVmZW5kLCBhbmQgaG9sZCBlYWNoIENvbnRyaWJ1dG9yIGhhcm1sZXNzIGZvciBhbnkgbGlhYmlsaXR5CiAgICAgIGluY3VycmVkIGJ5LCBvciBjbGFpbXMgYXNzZXJ0ZWQgYWdhaW5zdCwgc3VjaCBDb250cmlidXRvciBieSByZWFzb24KICAgICAgb2YgeW91ciBhY2NlcHRpbmcgYW55IHN1Y2ggd2FycmFudHkgb3IgYWRkaXRpb25hbCBsaWFiaWxpdHkuCgogICBFTkQgT0YgVEVSTVMgQU5EIENPTkRJVElPTlMKCiAgIEFQUEVORElYOiBIb3cgdG8gYXBwbHkgdGhlIEFwYWNoZSBMaWNlbnNlIHRvIHlvdXIgd29yay4KCiAgICAgIFRvIGFwcGx5IHRoZSBBcGFjaGUgTGljZW5zZSB0byB5b3VyIHdvcmssIGF0dGFjaCB0aGUgZm9sbG93aW5nCiAgICAgIGJvaWxlcnBsYXRlIG5vdGljZSwgd2l0aCB0aGUgZmllbGRzIGVuY2xvc2VkIGJ5IGJyYWNrZXRzICJbXSIKICAgICAgcmVwbGFjZWQgd2l0aCB5b3VyIG93biBpZGVudGlmeWluZyBpbmZvcm1hdGlvbi4gKERvbid0IGluY2x1ZGUKICAgICAgdGhlIGJyYWNrZXRzISkgIFRoZSB0ZXh0IHNob3VsZCBiZSBlbmNsb3NlZCBpbiB0aGUgYXBwcm9wcmlhdGUKICAgICAgY29tbWVudCBzeW50YXggZm9yIHRoZSBmaWxlIGZvcm1hdC4gV2UgYWxzbyByZWNvbW1lbmQgdGhhdCBhCiAgICAgIGZpbGUgb3IgY2xhc3MgbmFtZSBhbmQgZGVzY3JpcHRpb24gb2YgcHVycG9zZSBiZSBpbmNsdWRlZCBvbiB0aGUKICAgICAgc2FtZSAicHJpbnRlZCBwYWdlIiBhcyB0aGUgY29weXJpZ2h0IG5vdGljZSBmb3IgZWFzaWVyCiAgICAgIGlkZW50aWZpY2F0aW9uIHdpdGhpbiB0aGlyZC1wYXJ0eSBhcmNoaXZlcy4KCiAgIENvcHlyaWdodCBbeXl5eV0gW25hbWUgb2YgY29weXJpZ2h0IG93bmVyXQoKICAgTGljZW5zZWQgdW5kZXIgdGhlIEFwYWNoZSBMaWNlbnNlLCBWZXJzaW9uIDIuMCAodGhlICJMaWNlbnNlIik7CiAgIHlvdSBtYXkgbm90IHVzZSB0aGlzIGZpbGUgZXhjZXB0IGluIGNvbXBsaWFuY2Ugd2l0aCB0aGUgTGljZW5zZS4KICAgWW91IG1heSBvYnRhaW4gYSBjb3B5IG9mIHRoZSBMaWNlbnNlIGF0CgogICAgICAgaHR0cDovL3d3dy5hcGFjaGUub3JnL2xpY2Vuc2VzL0xJQ0VOU0UtMi4wCgogICBVbmxlc3MgcmVxdWlyZWQgYnkgYXBwbGljYWJsZSBsYXcgb3IgYWdyZWVkIHRvIGluIHdyaXRpbmcsIHNvZnR3YXJlCiAgIGRpc3RyaWJ1dGVkIHVuZGVyIHRoZSBMaWNlbnNlIGlzIGRpc3RyaWJ1dGVkIG9uIGFuICJBUyBJUyIgQkFTSVMsCiAgIFdJVEhPVVQgV0FSUkFOVElFUyBPUiBDT05ESVRJT05TIE9GIEFOWSBLSU5ELCBlaXRoZXIgZXhwcmVzcyBvciBpbXBsaWVkLgogICBTZWUgdGhlIExpY2Vuc2UgZm9yIHRoZSBzcGVjaWZpYyBsYW5ndWFnZSBnb3Zlcm5pbmcgcGVybWlzc2lvbnMgYW5kCiAgIGxpbWl0YXRpb25zIHVuZGVyIHRoZSBMaWNlbnNlLgo=")
EXPECTED_ENGINE_SHA256 = "bc8a54879ef02c7ea64b8b333d6a976f0ea65c4949149d01f463f23bccee653e"
EXPECTED_SCHEMA_SHA256 = "a82c89c1a2315b93f39775d8e025471a01b738647c9772658368ee6b1b6f4867"
EXPECTED_ENGINE_LICENSE_SHA256 = "c71d239df91726fc519c6eb72d318ec65820627232b2f796219e87dcf35d0ab4"
EXPECTED_ACTIONS_SHA256 = ['9d1caf79e29316809dbcb851fd4556d6b222cc392463a470ccf5ee978be13a6c', 'fc8e02da7431b976e4e49d948763e6f0f2be08f1702ca813188d88e990b1bc06']
EXPECTED_REWARDS = [94976.0, 94933.0]

def sha256(data):
    return hashlib.sha256(data).hexdigest()

assert sha256(ENGINE_SOURCE) == EXPECTED_ENGINE_SHA256
assert sha256(ENGINE_SCHEMA) == EXPECTED_SCHEMA_SHA256
assert sha256(ENGINE_LICENSE) == EXPECTED_ENGINE_LICENSE_SHA256
try:
    HOST_CORE_VERSION = importlib.metadata.version("kaggle-environments")
except importlib.metadata.PackageNotFoundError:
    HOST_CORE_VERSION = "distribution metadata unavailable"
print("Observed Kaggle host core before engine import:", HOST_CORE_VERSION)
SEED_HELPER_SHIM_INSTALLED = False
PINNED_GAME_DIR = Path(RUNTIME_CLEANUP.name) / "kaggriculture-engine"
PINNED_GAME_DIR.mkdir(parents=True, exist_ok=True)
ENGINE_FILE = PINNED_GAME_DIR / "kaggriculture.py"
SCHEMA_FILE = PINNED_GAME_DIR / "kaggriculture.json"
LICENSE_FILE = PINNED_GAME_DIR / "LICENSE"
ENGINE_FILE.write_bytes(ENGINE_SOURCE)
SCHEMA_FILE.write_bytes(ENGINE_SCHEMA)
LICENSE_FILE.write_bytes(ENGINE_LICENSE)

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    saved_out, saved_err = os.dup(1), os.dup(2)
    sink = os.open(os.devnull, os.O_WRONLY)
    try:
        os.dup2(sink, 1)
        os.dup2(sink, 2)
        from kaggle_environments import utils as _kaggle_utils
        if not hasattr(_kaggle_utils, "resolve_episode_seed"):
            # Copyright 2020 Kaggle Inc.; Apache-2.0. Exact source from kaggle_environments/utils.py.
            from typing import Any, Callable
            import random
            _VENDORED_SEED_HELPER_SOURCE = 'def resolve_episode_seed(\n    env: Any,\n    *,\n    config_key: str = "seed",\n    fallback: Callable[[], int] | None = None,\n) -> int:\n    """Resolve, scrub, and persist an episode seed for envs with hidden state.\n\n    Used by interpreters whose initial state depends on a seed that agents\n    must not be able to read (e.g. random maze layout, comet schedules,\n    weather rolls). The seed is taken from the first available source:\n    ``env.info["seed"]`` (preserved across re-initialization), then\n    ``configuration[config_key]``, then ``fallback()`` (defaulting to a random\n    31-bit int). The value is then cleared from ``configuration`` so agents\n    can\'t read it via the observation, and stored on ``env.info["seed"]`` so\n    it persists into the replay.\n    """\n    if not hasattr(env, "info") or env.info is None:\n        env.info = {}\n    seed = env.info.get("seed")\n    config = env.configuration\n    if seed is None:\n        seed = getattr(config, config_key, None)\n        if seed is None and isinstance(config, dict):\n            seed = config.get(config_key)\n    if seed is None:\n        seed = fallback() if fallback is not None else random.randrange(2**31)\n    try:\n        setattr(config, config_key, None)\n    except (AttributeError, TypeError):\n        config[config_key] = None\n    env.info["seed"] = seed\n    return seed'
            EXPECTED_SEED_HELPER_SHA256 = "15feb82a21849d8bb07d0bad806dc287992af24bc8d0370031f40c4b1b856a54"
            assert sha256(_VENDORED_SEED_HELPER_SOURCE.encode("utf-8")) == EXPECTED_SEED_HELPER_SHA256
            _seed_helper_namespace = {"Any": Any, "Callable": Callable, "random": random}
            exec(compile(_VENDORED_SEED_HELPER_SOURCE, "kaggle-utils-resolve-episode-seed", "exec"), _seed_helper_namespace)
            _kaggle_utils.resolve_episode_seed = _seed_helper_namespace["resolve_episode_seed"]
            SEED_HELPER_SHIM_INSTALLED = True
        engine_spec = importlib.util.spec_from_file_location("kaggriculture_pinned_1327", ENGINE_FILE)
        assert engine_spec is not None and engine_spec.loader is not None
        pinned_engine = importlib.util.module_from_spec(engine_spec)
        engine_spec.loader.exec_module(pinned_engine)
        from kaggle_environments import environments, make, register
        assert callable(make) and callable(register), "Installed Kaggle core lacks make/register APIs."
        register("kaggriculture", {
            "agents": pinned_engine.agents,
            "html_renderer": pinned_engine.html_renderer,
            "interpreter": pinned_engine.interpreter,
            "renderer": pinned_engine.renderer,
            "specification": pinned_engine.specification,
        })
    finally:
        os.dup2(saved_out, 1)
        os.dup2(saved_err, 2)
        os.close(saved_out)
        os.close(saved_err)
        os.close(sink)
registered = environments["kaggriculture"]
assert registered["interpreter"] is pinned_engine.interpreter
assert registered["specification"] == json.loads(ENGINE_SCHEMA)
if SEED_HELPER_SHIM_INSTALLED:
    print("Installed byte-verified Kaggle seed-helper compatibility shim:", EXPECTED_SEED_HELPER_SHA256)
print("Registered exact Kaggriculture engine SHA-256:", EXPECTED_ENGINE_SHA256)
print("Registered exact schema SHA-256:", EXPECTED_SCHEMA_SHA256)

In [4]:
import contextlib, hashlib, io, json, os, sys
from contextlib import contextmanager

@contextmanager
def silence_native_output():
    sys.stdout.flush()
    sys.stderr.flush()
    saved_out, saved_err = os.dup(1), os.dup(2)
    sink = os.open(os.devnull, os.O_WRONLY)
    try:
        os.dup2(sink, 1)
        os.dup2(sink, 2)
        yield
    finally:
        os.dup2(saved_out, 1)
        os.dup2(saved_err, 2)
        os.close(saved_out)
        os.close(saved_err)
        os.close(sink)

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()), silence_native_output():
    env = make("kaggriculture", configuration={"episodeSteps": 720, "seed": 29454100})
    steps = env.run([str(MAIN), str(CONTROL)])
assert len(steps) == 720
final = steps[-1]
statuses = [agent["status"] for agent in final]
rewards = [agent["reward"] for agent in final]
callbacks_per_agent = [sum(state[i].get("action") is not None for state in steps[1:]) for i in range(2)]
actions = [[row[i].get("action") for row in steps[1:]] for i in range(2)]
def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
actions_sha256 = [digest([digest(action) for action in seat]) for seat in actions]
assert statuses == ["DONE", "DONE"]
assert callbacks_per_agent == [719, 719]
assert all(isinstance(value, (int, float)) for value in rewards)
assert actions_sha256 == EXPECTED_ACTIONS_SHA256, "Pinned full-game action stream mismatch."
assert rewards == EXPECTED_REWARDS, "Pinned full-game rewards mismatch."
print("PASS: host core", HOST_CORE_VERSION, "used the exact pinned Kaggriculture source/schema.")
print("PASS: 720 states; DONE/DONE; 719 callbacks per seat.")
print("PASS: action SHA-256", actions_sha256, "rewards", rewards)
RUNTIME_CLEANUP.cleanup()
assert not Path(str(MAIN)).exists() and not Path(str(NOTICE)).exists()
print("PASS: temporary policy, game source, schema and license removed; submission archive remains.")